# 06. 코로나19 전후 택시 수요 변화 분석

티머니 STIS 택시 데이터(D012)를 활용하여 코로나19 전후 택시 수요 변화를 분석한다.

**기간 구분:**
| 구분 | 기간 |
|------|------|
| Pre-COVID | 2018.01 ~ 2020.01 |
| COVID-초기 | 2020.02 ~ 2020.12 |
| COVID-중기 | 2021.01 ~ 2021.12 |
| COVID-후기 | 2022.01 ~ 2022.12 |
| Post-COVID | 2023.01 ~ 2026.04 |

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

# 폰트 설정 (Windows)
plt.rcParams['font.family'] = 'Malgun Gothic'
# Mac: plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

In [2]:
# 데이터 로드
df = pd.read_csv(r"C:\Users\admin\Desktop\작업 폴더\tmoney\DC_TBYXD012.csv")
print(f'전체 건수: {len(df):,}')
df.head()

전체 건수: 60,000,000


,COL_DTIME,TRANSP_BIZR_ID,TAXI_VEHC_ID,DRIVER_ID,COMPX_PAY_YN,TR_TYPE,FARE_CLASS_CD,CARD_RIDE_FARE,CASH_RIDE_FARE,CALL_FARE,...,RIDE_SL_DIST,UUID,RIDE_SEQ_NO,DELAY_PUSH_YN,PLTF_FEE_AMT,FEE_MODEL_CLASS_CD,RESERVED,SYNC_DTIME,RIDE_POS_Y_ENC,ALIGHT_POS_Y_ENC
0,20210707005346,TB00069500,V000000322799,D129581,N,1,3,0,6600,0,...,2566,00000000000000000000000000000000,656,N,500,3,NaN,20210707014409,0,000000000000000000000000
1,20220101032136,TB00067935,V000000202137,D99776,N,3,1,0,10100,3000,...,1709,00000000000000000000000000000001,199,N,1000,1,NaN,20220101051905,0,000000000000000000000001
2,20250911051822,TB00084292,V000000263067,D202590,N,1,2,8300,0,0,...,3354,00000000000000000000000000000002,723,N,1500,1,NaN,20250911070335,0,000000000000000000000002
3,20190523081834,TB00065265,V000000026498,D267897,N,1,3,5800,0,0,...,1248,00000000000000000000000000000003,405,N,0,3,NaN,20190523084801,0,000000000000000000000003
4,20220830201134,TB00039357,V000000124618,D135234,N,2,3,6100,0,1000,...,2690,00000000000000000000000000000004,391,N,0,2,NaN,20220830202236,0,000000000000000000000004


In [ ]:
# 승차시간 파싱 및 파생 컬럼 생성
df['RIDE_DT'] = pd.to_datetime(df['RIDE_DTIME'], format='%Y%m%d%H%M%S')
df['YM'] = df['RIDE_DT'].dt.to_period('M')
df['YEAR'] = df['RIDE_DT'].dt.year
df['MONTH'] = df['RIDE_DT'].dt.month
df['HOUR'] = df['RIDE_DT'].dt.hour
df['DOW'] = df['RIDE_DT'].dt.dayofweek  # 0=월 6=일

# 기간 구분 라벨
def assign_period(dt):
    if dt < pd.Timestamp('2020-02-01'):
        return 'Pre-COVID'
    elif dt < pd.Timestamp('2021-01-01'):
        return 'COVID-초기'
    elif dt < pd.Timestamp('2022-01-01'):
        return 'COVID-중기'
    elif dt < pd.Timestamp('2023-01-01'):
        return 'COVID-후기'
    else:
        return 'Post-COVID'

df['PERIOD'] = df['RIDE_DT'].apply(assign_period)

period_order = ['Pre-COVID', 'COVID-초기', 'COVID-중기', 'COVID-후기', 'Post-COVID']
df['PERIOD'] = pd.Categorical(df['PERIOD'], categories=period_order, ordered=True)

print('기간별 건수:')
print(df['PERIOD'].value_counts().sort_index())

## 1. 월별 총 승차건수 시계열 (기간별 색상 구분)

In [ ]:
monthly = df.groupby('YM').size().reset_index(name='COUNT')
monthly['YM_STR'] = monthly['YM'].astype(str)
monthly['YM_DT'] = pd.to_datetime(monthly['YM_STR'])

# 기간별 색상 매핑
period_colors = {
    'Pre-COVID': '#2196F3',
    'COVID-초기': '#F44336',
    'COVID-중기': '#FF9800',
    'COVID-후기': '#FFC107',
    'Post-COVID': '#4CAF50'
}

def get_period_color(dt):
    if dt < pd.Timestamp('2020-02-01'):
        return period_colors['Pre-COVID']
    elif dt < pd.Timestamp('2021-01-01'):
        return period_colors['COVID-초기']
    elif dt < pd.Timestamp('2022-01-01'):
        return period_colors['COVID-중기']
    elif dt < pd.Timestamp('2023-01-01'):
        return period_colors['COVID-후기']
    else:
        return period_colors['Post-COVID']

monthly['COLOR'] = monthly['YM_DT'].apply(get_period_color)

fig, ax = plt.subplots(figsize=(16, 5))
ax.bar(monthly['YM_DT'], monthly['COUNT'], color=monthly['COLOR'], width=25)

# 범례
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=p) for p, c in period_colors.items()]
ax.legend(handles=legend_elements, loc='upper right')

ax.set_title('월별 총 승차건수 (기간별 색상 구분)', fontsize=14)
ax.set_xlabel('월')
ax.set_ylabel('승차건수')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 2. 시간대별 수요 프로파일 비교 (Pre vs COVID-초기 vs Post)

In [ ]:
compare_periods = ['Pre-COVID', 'COVID-초기', 'Post-COVID']
compare_colors = ['#2196F3', '#F44336', '#4CAF50']

fig, ax = plt.subplots(figsize=(12, 5))

for period, color in zip(compare_periods, compare_colors):
    subset = df[df['PERIOD'] == period]
    # 월 수로 나누어 월 평균 건수로 비교
    n_months = subset['YM'].nunique()
    hourly = subset.groupby('HOUR').size() / max(n_months, 1)
    ax.plot(hourly.index, hourly.values, marker='o', label=period, color=color, linewidth=2)

ax.set_title('시간대별 수요 프로파일 비교 (월 평균)', fontsize=14)
ax.set_xlabel('시간대')
ax.set_ylabel('월 평균 승차건수')
ax.set_xticks(range(24))
ax.legend()
ax.grid(True, alpha=0.3)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.tight_layout()
plt.show()

## 3. 요일별 수요 패턴 변화 (Pre vs Post)

In [ ]:
dow_labels = ['월', '화', '수', '목', '금', '토', '일']

fig, ax = plt.subplots(figsize=(10, 5))

for period, color in [('Pre-COVID', '#2196F3'), ('Post-COVID', '#4CAF50')]:
    subset = df[df['PERIOD'] == period]
    n_weeks = (subset['RIDE_DT'].max() - subset['RIDE_DT'].min()).days / 7
    dow_counts = subset.groupby('DOW').size() / max(n_weeks, 1)
    ax.plot(dow_counts.index, dow_counts.values, marker='o', label=period, color=color, linewidth=2)

ax.set_title('요일별 수요 패턴 변화 (주 평균)', fontsize=14)
ax.set_xlabel('요일')
ax.set_ylabel('주 평균 승차건수')
ax.set_xticks(range(7))
ax.set_xticklabels(dow_labels)
ax.legend()
ax.grid(True, alpha=0.3)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.tight_layout()
plt.show()

## 4. 행정동별 수요 변화율 Top/Bottom 10

In [ ]:
# Pre-COVID vs Post-COVID 행정동별 월 평균 승차건수 비교
pre = df[df['PERIOD'] == 'Pre-COVID']
post = df[df['PERIOD'] == 'Post-COVID']

pre_months = pre['YM'].nunique()
post_months = post['YM'].nunique()

pre_area = pre.groupby('RIDE_A_CD').size() / max(pre_months, 1)
post_area = post.groupby('RIDE_A_CD').size() / max(post_months, 1)

area_change = pd.DataFrame({'Pre': pre_area, 'Post': post_area}).dropna()
# 건수가 너무 적은 행정동 제외
area_change = area_change[area_change['Pre'] >= 10]
area_change['변화율(%)'] = (area_change['Post'] - area_change['Pre']) / area_change['Pre'] * 100

top10 = area_change.nlargest(10, '변화율(%)')
bot10 = area_change.nsmallest(10, '변화율(%)')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Top 10
axes[0].barh(top10.index.astype(str), top10['변화율(%)'], color='#4CAF50')
axes[0].set_title('수요 증가율 Top 10 행정동', fontsize=13)
axes[0].set_xlabel('변화율 (%)')
axes[0].invert_yaxis()

# Bottom 10
axes[1].barh(bot10.index.astype(str), bot10['변화율(%)'], color='#F44336')
axes[1].set_title('수요 감소율 Bottom 10 행정동', fontsize=13)
axes[1].set_xlabel('변화율 (%)')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

print('\n--- 수요 증가율 Top 10 ---')
print(top10[['Pre', 'Post', '변화율(%)']].round(1).to_string())
print('\n--- 수요 감소율 Bottom 10 ---')
print(bot10[['Pre', 'Post', '변화율(%)']].round(1).to_string())

## 5. 평균 이동거리 / 요금 변화 추이

In [ ]:
monthly_stats = df.groupby('YM').agg(
    평균거리=('RIDE_DIST', 'mean'),
    평균요금=('PAY_AMT', 'mean')
).reset_index()
monthly_stats['YM_DT'] = pd.to_datetime(monthly_stats['YM'].astype(str))

fig, ax1 = plt.subplots(figsize=(16, 5))

color1 = '#1976D2'
color2 = '#E64A19'

ax1.plot(monthly_stats['YM_DT'], monthly_stats['평균거리'], color=color1, linewidth=1.5, label='평균 이동거리(m)')
ax1.set_xlabel('월')
ax1.set_ylabel('평균 이동거리 (m)', color=color1)
ax1.tick_params(axis='y', labelcolor=color1)

ax2 = ax1.twinx()
ax2.plot(monthly_stats['YM_DT'], monthly_stats['평균요금'], color=color2, linewidth=1.5, label='평균 요금(원)')
ax2.set_ylabel('평균 요금 (원)', color=color2)
ax2.tick_params(axis='y', labelcolor=color2)

# 기간 구분 배경
period_ranges = [
    ('2018-01-01', '2020-01-31', '#2196F3', 0.07),
    ('2020-02-01', '2020-12-31', '#F44336', 0.07),
    ('2021-01-01', '2021-12-31', '#FF9800', 0.07),
    ('2022-01-01', '2022-12-31', '#FFC107', 0.07),
    ('2023-01-01', '2026-04-30', '#4CAF50', 0.07),
]
for s, e, c, a in period_ranges:
    ax1.axvspan(pd.Timestamp(s), pd.Timestamp(e), color=c, alpha=a)

ax1.set_title('월별 평균 이동거리 / 요금 변화 추이', fontsize=14)
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 6. 심야(23~04시) vs 주간 수요 비율 변화

In [ ]:
df['IS_NIGHT'] = df['HOUR'].isin([23, 0, 1, 2, 3, 4])

night_ratio = df.groupby('YM')['IS_NIGHT'].mean().reset_index()
night_ratio.columns = ['YM', 'NIGHT_RATIO']
night_ratio['YM_DT'] = pd.to_datetime(night_ratio['YM'].astype(str))
night_ratio['NIGHT_PCT'] = night_ratio['NIGHT_RATIO'] * 100

fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(night_ratio['YM_DT'], night_ratio['NIGHT_PCT'], color='#5C6BC0', linewidth=1.5, marker='.')

for s, e, c, a in period_ranges:
    ax.axvspan(pd.Timestamp(s), pd.Timestamp(e), color=c, alpha=0.07)

ax.set_title('월별 심야(23~04시) 수요 비율 변화', fontsize=14)
ax.set_xlabel('월')
ax.set_ylabel('심야 비율 (%)')
ax.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 7. 기간별 요약 테이블

In [ ]:
summary = df.groupby('PERIOD').agg(
    총건수=('RIDE_DT', 'size'),
    월수=('YM', 'nunique'),
    평균요금=('PAY_AMT', 'mean'),
    평균거리=('RIDE_DIST', 'mean'),
    심야비율=('IS_NIGHT', 'mean'),
).reset_index()

summary['월평균건수'] = (summary['총건수'] / summary['월수']).astype(int)
summary['심야비율(%)'] = (summary['심야비율'] * 100).round(1)
summary['평균요금'] = summary['평균요금'].round(0).astype(int)
summary['평균거리'] = summary['평균거리'].round(0).astype(int)

# Pre-COVID 대비 변화율
pre_avg = summary.loc[summary['PERIOD'] == 'Pre-COVID', '월평균건수'].values[0]
summary['수요변화율(%)'] = ((summary['월평균건수'] - pre_avg) / pre_avg * 100).round(1)

display_cols = ['PERIOD', '총건수', '월수', '월평균건수', '수요변화율(%)', '평균요금', '평균거리', '심야비율(%)']
summary_display = summary[display_cols].copy()
summary_display.columns = ['기간', '총건수', '월수', '월평균건수', '수요변화율(%)', '평균요금(원)', '평균거리(m)', '심야비율(%)']

print('=== 코로나19 전후 택시 수요 요약 ===')
print(summary_display.to_string(index=False))

## 요약

- Pre-COVID 대비 COVID-초기 수요 급감 여부 및 규모를 월별 시계열로 확인
- 시간대별 프로파일에서 심야/출퇴근 시간 수요 구조 변화 파악
- 요일별 패턴에서 평일/주말 수요 비중 변화 확인
- 행정동별 수요 변화율로 지역별 회복/위축 차이 식별
- 평균 이동거리/요금 추이를 통해 이용 행태 변화 파악
- 심야 수요 비율 변화로 영업시간대 구조 변화 확인